# 15. What Is a Geomagnetic Storm? — Implementation\n# 지자기 폭풍이란 무엇인가? — 구현\n\n**Paper**: Gonzalez, W. D., et al. (1994), *J. Geophys. Res.*, 99(A4), 5771–5792\n\nThis notebook implements key quantitative frameworks from the paper:\n이 노트북은 논문의 핵심 정량적 프레임워크를 구현합니다:\n\n1. **Burton Equation** — Dst prediction from solar wind parameters / 태양풍 매개변수로부터 Dst 예측\n2. **Energy Coupling Functions** — Comparison of Table 2 functions / Table 2 결합 함수 비교\n3. **DPS Relation** — Ring current energy ↔ Dst / Ring current 에너지 ↔ Dst\n4. **Storm Classification** — Dst-based intensity categories / Dst 기반 강도 분류\n5. **Seasonal Distribution** — Russell-McPherron effect visualization / Russell-McPherron 효과 시각화

In [ ]:
import numpy as np\nimport matplotlib.pyplot as plt\nfrom matplotlib.patches import FancyBboxPatch\nfrom scipy.integrate import solve_ivp\n\nplt.rcParams.update({\n    "figure.figsize": (12, 6),\n    "font.size": 12,\n    "axes.grid": True,\n    "grid.alpha": 0.3,\n})

## 1. Burton Equation: Dst Prediction / Burton 방정식: Dst 예측\n\nThe Burton equation (Burton et al., 1975) describes the energy balance of the ring current:\nBurton 방정식은 ring current의 에너지 균형을 기술합니다:\n\n$$\\frac{dDst^*}{dt} = Q(t) - \\frac{Dst^*}{\\tau}$$\n\nwhere:\n- $Dst^* = Dst - b\\sqrt{p} + c$ is the pressure-corrected Dst / 압력 보정된 Dst\n- $Q(t)$ is the energy injection rate, proportional to $VB_s$ / 에너지 주입률\n- $\\tau$ is the ring current decay time / ring current 소산 시간상수\n\nWe simulate a synthetic intense storm driven by a magnetic cloud with rotating $B_z$.\n회전하는 $B_z$를 가진 magnetic cloud에 의한 합성 intense storm을 시뮬레이션합니다.

In [ ]:
def burton_equation(t: float, dst_star: np.ndarray, t_hours: np.ndarray,\n                    q_values: np.ndarray, tau: float) -> list[float]:\n    """Right-hand side of the Burton equation.\n\n    Args:\n        t: Current time in hours.\n        dst_star: Current pressure-corrected Dst [nT].\n        t_hours: Time array for Q interpolation.\n        q_values: Energy injection rate array [nT/h].\n        tau: Ring current decay time [hours].\n\n    Returns:\n        Time derivative of Dst* [nT/h].\n    """\n    q = np.interp(t, t_hours, q_values)\n    return [q - dst_star[0] / tau]\n\n\ndef compute_bs(bz: np.ndarray) -> np.ndarray:\n    """Extract southward IMF component.\n\n    Args:\n        bz: IMF Bz component [nT]. Negative = southward.\n\n    Returns:\n        Bs: southward component (positive when Bz < 0, zero otherwise).\n    """\n    return np.where(bz < 0, -bz, 0.0)\n\n\ndef injection_rate(v: np.ndarray, bs: np.ndarray,\n                   d: float = -1.5e-3, e_crit: float = 0.5) -> np.ndarray:\n    """Compute energy injection rate Q from Burton et al. (1975).\n\n    Args:\n        v: Solar wind speed [km/s].\n        bs: Southward IMF component [nT].\n        d: Injection coefficient [nT per (km/s * nT * h)].\n        e_crit: Electric field threshold [mV/m].\n\n    Returns:\n        Q: Energy injection rate [nT/h].\n    """\n    ey = v * bs * 1e-3  # Convert to mV/m\n    q = np.where(ey > e_crit, d * (ey - e_crit) * 1e3, 0.0)\n    return q

### 1.1 Synthetic Magnetic Cloud Storm / 합성 Magnetic Cloud 폭풍\n\nWe create a synthetic storm scenario mimicking a magnetic cloud passage:\n자기 구름(magnetic cloud) 통과를 모사하는 합성 폭풍 시나리오를 생성합니다:\n\n- **SSC at t=0**: Interplanetary shock arrival, dynamic pressure jump\n- **Sheath (0–6 h)**: Compressed, turbulent IMF with intermittent southward $B_z$\n- **Magnetic cloud (6–24 h)**: Smooth rotation of $B_z$ from north to south (SN type)\n- **Recovery (24–72 h)**: $B_z$ returns to quiet; ring current decays

In [ ]:
# --- Synthetic solar wind parameters for a magnetic cloud storm ---\ndt = 0.1  # Time resolution [hours]\nt = np.arange(0, 72, dt)\n\n# Solar wind speed: jump at shock, elevated during cloud\nv_sw = np.full_like(t, 400.0)  # Quiet: 400 km/s\nv_sw[(t >= 0) & (t < 6)] = 550.0   # Sheath: compressed, fast\nv_sw[(t >= 6) & (t < 24)] = 500.0  # Magnetic cloud: moderately fast\nv_sw[(t >= 24)] = 420.0            # Post-cloud: gradual return\n\n# IMF Bz: sheath turbulence + smooth cloud rotation (SN → NS type)\nnp.random.seed(42)\nbz = np.full_like(t, 2.0)  # Quiet: +2 nT (slightly northward)\n\n# Sheath: turbulent, intermittent southward\nsheath_mask = (t >= 0) & (t < 6)\nbz[sheath_mask] = 5 * np.sin(2 * np.pi * t[sheath_mask] / 2) + \\\n                  3 * np.random.randn(sheath_mask.sum())\n\n# Magnetic cloud: smooth rotation from +15 nT (north) to -20 nT (south)\ncloud_mask = (t >= 6) & (t < 24)\nt_cloud = t[cloud_mask] - 6\nbz[cloud_mask] = 15 * np.cos(np.pi * t_cloud / 18)  # Rotates N → S\n\n# Dynamic pressure\np_dyn = np.full_like(t, 2.0)  # Quiet: 2 nPa\np_dyn[(t >= 0) & (t < 6)] = 10.0   # Shock compression\np_dyn[(t >= 6) & (t < 24)] = 5.0   # Cloud: elevated\np_dyn[(t >= 24)] = 2.5\n\n# Compute Bs and Q\nbs = compute_bs(bz)\nq = injection_rate(v_sw, bs)\n\n# Solve Burton equation\ntau = 7.7  # Decay time [hours]\nsol = solve_ivp(\n    burton_equation, [t[0], t[-1]], [0.0],\n    args=(t, q, tau), t_eval=t, method="RK45", max_step=0.1\n)\ndst_star = sol.y[0]\n\n# Convert Dst* back to observed Dst: Dst = Dst* + b*sqrt(p) - c\nb_coeff = 15.8  # nT/nPa^(1/2)\nc_coeff = 20.0  # nT\ndst_observed = dst_star + b_coeff * np.sqrt(p_dyn) - c_coeff\n\nprint(f"Peak Dst*: {dst_star.min():.1f} nT at t = {t[np.argmin(dst_star)]:.1f} h")\nprint(f"Peak observed Dst: {dst_observed.min():.1f} nT at t = {t[np.argmin(dst_observed)]:.1f} h")

In [ ]:
# --- Multi-panel storm plot (similar to Figure 1 & 3 in the paper) ---\nfig, axes = plt.subplots(5, 1, figsize=(14, 16), sharex=True)\n\n# Panel 1: Solar wind speed\naxes[0].plot(t, v_sw, "b-", linewidth=1.5)\naxes[0].set_ylabel("V [km/s]")\naxes[0].set_title("Synthetic Magnetic Cloud Storm — Burton Equation Simulation\\n"\n                   "합성 Magnetic Cloud 폭풍 — Burton 방정식 시뮬레이션")\naxes[0].set_ylim(350, 600)\n\n# Panel 2: IMF Bz\naxes[1].plot(t, bz, "r-", linewidth=1)\naxes[1].axhline(0, color="k", linewidth=0.5)\naxes[1].fill_between(t, bz, 0, where=(bz < 0), alpha=0.3, color="red",\n                     label="Southward $B_z$")\naxes[1].set_ylabel("IMF $B_z$ [nT]")\naxes[1].legend(loc="upper right")\n\n# Panel 3: Energy injection rate Q\naxes[2].plot(t, q, "g-", linewidth=1.5)\naxes[2].set_ylabel("Q [nT/h]")\naxes[2].set_ylim(min(q.min() * 1.1, -5), 5)\n\n# Panel 4: Dynamic pressure\naxes[3].plot(t, p_dyn, "m-", linewidth=1.5)\naxes[3].set_ylabel("$P_{dyn}$ [nPa]")\n\n# Panel 5: Dst\naxes[4].plot(t, dst_star, "b-", linewidth=2, label="$Dst^*$ (corrected)")\naxes[4].plot(t, dst_observed, "k--", linewidth=1.5, label="$Dst$ (observed)")\naxes[4].axhline(-30, color="orange", linestyle=":", label="Weak threshold (−30 nT)")\naxes[4].axhline(-50, color="darkorange", linestyle=":", label="Moderate threshold (−50 nT)")\naxes[4].axhline(-100, color="red", linestyle=":", label="Intense threshold (−100 nT)")\naxes[4].set_ylabel("Dst [nT]")\naxes[4].set_xlabel("Time [hours]")\naxes[4].legend(loc="lower right", fontsize=10)\n\n# Add phase annotations\nfor ax in axes:\n    ax.axvspan(0, 6, alpha=0.08, color="yellow", label="")\n    ax.axvspan(6, 24, alpha=0.08, color="cyan", label="")\n\n# Phase labels on top panel\naxes[0].text(3, 580, "Sheath\\n피복 영역", ha="center", fontsize=10,\n             bbox=dict(boxstyle="round", facecolor="yellow", alpha=0.5))\naxes[0].text(15, 580, "Magnetic Cloud\\n자기 구름", ha="center", fontsize=10,\n             bbox=dict(boxstyle="round", facecolor="cyan", alpha=0.5))\naxes[0].text(48, 580, "Recovery\\n회복상", ha="center", fontsize=10,\n             bbox=dict(boxstyle="round", facecolor="white", alpha=0.5))\n\nplt.tight_layout()\nplt.savefig("15_gonzalez_1994_fig1_burton_simulation.png", dpi=150, bbox_inches="tight")\nplt.show()

### 1.2 Effect of Decay Time $\\tau$ / 소산 시간 $\\tau$의 영향\n\nThe paper notes that $\\tau$ is not a fixed constant — it varies from ~0.5 h (intense storms) to ~10 h (recovery). We compare different $\\tau$ values.\n논문은 $\\tau$가 고정 상수가 아님을 강조합니다. 다양한 $\\tau$ 값의 영향을 비교합니다.

In [ ]:
# --- Compare different tau values ---\ntau_values = [3.0, 5.0, 7.7, 10.0, 15.0]\ncolors_tau = plt.cm.viridis(np.linspace(0.1, 0.9, len(tau_values)))\n\nfig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True)\n\nfor tau_val, color in zip(tau_values, colors_tau):\n    sol_tau = solve_ivp(\n        burton_equation, [t[0], t[-1]], [0.0],\n        args=(t, q, tau_val), t_eval=t, method="RK45", max_step=0.1\n    )\n    ax1.plot(t, sol_tau.y[0], color=color, linewidth=1.5,\n             label=f"$\\\\tau$ = {tau_val} h (peak: {sol_tau.y[0].min():.0f} nT)")\n\nax1.axhline(-100, color="red", linestyle=":", alpha=0.5)\nax1.set_ylabel("$Dst^*$ [nT]")\nax1.set_title("Effect of Decay Time $\\\\tau$ on Storm Intensity\\n"\n              "소산 시간 $\\\\tau$가 폭풍 강도에 미치는 영향")\nax1.legend(loc="lower right")\n\n# Show recovery comparison (zoom into recovery phase)\nfor tau_val, color in zip(tau_values, colors_tau):\n    sol_tau = solve_ivp(\n        burton_equation, [t[0], t[-1]], [0.0],\n        args=(t, q, tau_val), t_eval=t, method="RK45", max_step=0.1\n    )\n    # Normalize to peak for recovery comparison\n    peak_idx = np.argmin(sol_tau.y[0])\n    if peak_idx < len(t) - 1:\n        recovery_t = t[peak_idx:] - t[peak_idx]\n        recovery_dst = sol_tau.y[0][peak_idx:] / sol_tau.y[0][peak_idx]\n        ax2.plot(recovery_t, recovery_dst, color=color, linewidth=1.5,\n                 label=f"$\\\\tau$ = {tau_val} h")\n\n# Theoretical exponential decay\nt_theory = np.linspace(0, 50, 100)\nax2.plot(t_theory, np.exp(-t_theory / 7.7), "k--", linewidth=1,\n         label="$e^{-t/7.7}$ (Burton)")\n\nax2.set_xlabel("Time since peak [hours]")\nax2.set_ylabel("Normalized $Dst^*$")\nax2.set_title("Recovery Phase Comparison / 회복상 비교")\nax2.set_xlim(0, 50)\nax2.legend()\n\nplt.tight_layout()\nplt.savefig("15_gonzalez_1994_fig2_tau_comparison.png", dpi=150, bbox_inches="tight")\nplt.show()

## 2. Energy Coupling Functions (Table 2) / 에너지 결합 함수\n\nThe paper summarizes commonly used coupling functions (Table 2). We implement and compare several:\n논문은 Table 2에서 다양한 coupling function을 정리합니다. 주요 함수들을 구현하고 비교합니다:\n\n| Function | Formula | Reference |\n|---|---|---|\n| $VB_s$ | $E_y = VB_s$ | Burton et al. [1975] |\n| $\\epsilon$ | $vL_0^2 B^2 \\sin^4(\\theta/2)$ | Perreault & Akasofu [1978] |\n| $F_A$ | $p^{-1/3}V B^2 \\sin^4(\\theta/2)$ | Vasyliunas et al. [1982] |\n| $VB_T\\sin^2(\\theta/2)$ | Half-wave rectifier variant | Kan & Lee [1979] |

In [ ]:
def coupling_vbs(v: np.ndarray, bz: np.ndarray, by: np.ndarray) -> np.ndarray:\n    """VBs coupling function (Burton et al., 1975).\n\n    Args:\n        v: Solar wind speed [km/s].\n        bz: IMF Bz [nT].\n        by: IMF By [nT].\n\n    Returns:\n        Electric field Ey [mV/m].\n    """\n    bs = np.where(bz < 0, -bz, 0.0)\n    return v * bs * 1e-3  # mV/m\n\n\ndef coupling_epsilon(v: np.ndarray, bz: np.ndarray, by: np.ndarray,\n                     l0_re: float = 7.0) -> np.ndarray:\n    """Perreault-Akasofu epsilon function.\n\n    Args:\n        v: Solar wind speed [km/s].\n        bz: IMF Bz [nT].\n        by: IMF By [nT].\n        l0_re: Scale length in Earth radii.\n\n    Returns:\n        Epsilon [W].\n    """\n    re = 6.371e6  # Earth radius [m]\n    l0 = l0_re * re  # [m]\n    b_total = np.sqrt(by**2 + bz**2) * 1e-9  # [T]\n    theta = np.arctan2(by, bz)  # Clock angle\n    v_ms = v * 1e3  # [m/s]\n    return v_ms * l0**2 * b_total**2 * np.sin(theta / 2)**4\n\n\ndef coupling_fa(v: np.ndarray, bz: np.ndarray, by: np.ndarray,\n                rho: np.ndarray) -> np.ndarray:\n    """Vasyliunas et al. (1982) F_A coupling function.\n\n    Args:\n        v: Solar wind speed [km/s].\n        bz: IMF Bz [nT].\n        by: IMF By [nT].\n        rho: Solar wind density [cm^-3].\n\n    Returns:\n        F_A [arbitrary units for comparison].\n    """\n    bt = np.sqrt(by**2 + bz**2)\n    theta = np.arctan2(by, bz)\n    p_dyn = 1.67e-6 * rho * (v * 1e3)**2  # Rough dynamic pressure\n    return p_dyn**(-1/3) * v * bt**2 * np.sin(theta / 2)**4\n\n\ndef coupling_kan_lee(v: np.ndarray, bz: np.ndarray,\n                     by: np.ndarray) -> np.ndarray:\n    """Kan & Lee (1979) coupling: VBT*sin^2(theta/2).\n\n    Args:\n        v: Solar wind speed [km/s].\n        bz: IMF Bz [nT].\n        by: IMF By [nT].\n\n    Returns:\n        Coupling value [km/s * nT].\n    """\n    bt = np.sqrt(by**2 + bz**2)\n    theta = np.arctan2(by, bz)\n    return v * bt * np.sin(theta / 2)**2

In [ ]:
# --- Compute and compare coupling functions for the synthetic storm ---\n# Add By component for coupling functions that need it\nby = np.full_like(t, 0.0)\nby[sheath_mask] = 3 * np.sin(2 * np.pi * t[sheath_mask] / 3) + \\\n                  2 * np.random.randn(sheath_mask.sum())\nby[cloud_mask] = 8 * np.sin(np.pi * t_cloud / 18)  # Cloud By rotation\n\n# Density for F_A\nrho = np.full_like(t, 5.0)  # cm^-3\nrho[(t >= 0) & (t < 6)] = 20.0  # Sheath compression\nrho[(t >= 6) & (t < 24)] = 8.0  # Cloud\n\n# Compute all coupling functions\ncf_vbs = coupling_vbs(v_sw, bz, by)\ncf_eps = coupling_epsilon(v_sw, bz, by)\ncf_fa = coupling_fa(v_sw, bz, by, rho)\ncf_kl = coupling_kan_lee(v_sw, bz, by)\n\n# Normalize each to [0, 1] for comparison\ndef normalize(x):\n    xmin, xmax = x.min(), x.max()\n    if xmax - xmin == 0:\n        return np.zeros_like(x)\n    return (x - xmin) / (xmax - xmin)\n\nfig, axes = plt.subplots(5, 1, figsize=(14, 14), sharex=True)\n\naxes[0].plot(t, bz, "r-", linewidth=1)\naxes[0].fill_between(t, bz, 0, where=(bz < 0), alpha=0.3, color="red")\naxes[0].set_ylabel("IMF $B_z$ [nT]")\naxes[0].set_title("Coupling Function Comparison (Table 2)\\n"\n                   "결합 함수 비교 (Table 2)")\n\naxes[1].plot(t, cf_vbs, "b-", linewidth=1.5)\naxes[1].set_ylabel("$VB_s$ [mV/m]")\naxes[1].text(0.02, 0.85, "Burton et al. [1975]", transform=axes[1].transAxes,\n             fontsize=10, bbox=dict(facecolor="lightblue", alpha=0.7))\n\naxes[2].plot(t, cf_eps * 1e-12, "g-", linewidth=1.5)  # TW\naxes[2].set_ylabel("$\\\\epsilon$ [TW]")\naxes[2].text(0.02, 0.85, "Perreault & Akasofu [1978]", transform=axes[2].transAxes,\n             fontsize=10, bbox=dict(facecolor="lightgreen", alpha=0.7))\n\naxes[3].plot(t, cf_kl, "m-", linewidth=1.5)\naxes[3].set_ylabel("$VB_T\\\\sin^2(\\\\theta/2)$")\naxes[3].text(0.02, 0.85, "Kan & Lee [1979]", transform=axes[3].transAxes,\n             fontsize=10, bbox=dict(facecolor="plum", alpha=0.7))\n\n# Normalized overlay\nfor label, cf, color in [\n    ("$VB_s$", cf_vbs, "blue"),\n    ("$\\\\epsilon$", cf_eps, "green"),\n    ("$VB_T\\\\sin^2$", cf_kl, "purple"),\n]:\n    axes[4].plot(t, normalize(cf), color=color, linewidth=1.5,\n                 label=label, alpha=0.8)\naxes[4].set_ylabel("Normalized")\naxes[4].set_xlabel("Time [hours]")\naxes[4].legend(loc="upper right")\naxes[4].set_title("Normalized Comparison / 정규화 비교")\n\nfor ax in axes:\n    ax.axvspan(0, 6, alpha=0.05, color="yellow")\n    ax.axvspan(6, 24, alpha=0.05, color="cyan")\n\nplt.tight_layout()\nplt.savefig("15_gonzalez_1994_fig3_coupling_functions.png", dpi=150, bbox_inches="tight")\nplt.show()

## 3. DPS Relation: Ring Current Energy ↔ Dst / DPS 관계: 환전류 에너지 ↔ Dst\n\nThe Dessler-Parker-Sckopke relation connects the Dst depression directly to ring current particle energy:\nDPS 관계식은 Dst 감소와 ring current 입자 에너지를 직접 연결합니다:\n\n$$\\frac{Dst^*}{B_0} \\approx \\frac{2E(t)}{3E_m}$$\n\nWe visualize this relationship and compute ring current energy for historical storm levels.\n이 관계를 시각화하고, 역사적 폭풍 수준에 대한 ring current 에너지를 계산합니다.

In [ ]:
# --- DPS Relation ---\nB0 = 31000  # Equatorial surface field [nT]\nEm = 8e24   # Total dipole field energy [ergs]\nEm_joules = Em * 1e-7  # Convert to Joules: 8e17 J\n\ndef dst_to_ring_current_energy(dst_star: float) -> float:\n    """Convert Dst* to ring current energy using DPS relation.\n\n    Args:\n        dst_star: Pressure-corrected Dst [nT].\n\n    Returns:\n        Ring current energy [Joules].\n    """\n    return 1.5 * abs(dst_star) / B0 * Em_joules\n\n# Historical storm levels\nstorm_levels = {\n    "Quiet (Dst = 0)": 0,\n    "Weak (Dst = -30 nT)": -30,\n    "Moderate (Dst = -75 nT)": -75,\n    "Intense (Dst = -150 nT)": -150,\n    "Super-intense (Dst = -300 nT)": -300,\n    "Carrington 1859 (est. -850 nT)": -850,\n    "March 1989 Quebec (-589 nT)": -589,\n    "Halloween 2003 (-422 nT)": -422,\n}\n\nprint("DPS Relation: Ring Current Energy for Historical Storms")\nprint("DPS 관계: 역사적 폭풍에 대한 Ring Current 에너지")\nprint("=" * 70)\nprint(f"{'Storm':40s} {'Dst [nT]':>10s} {'Energy [PJ]':>12s} {'E/Em':>10s}")\nprint("-" * 70)\nfor name, dst in storm_levels.items():\n    energy = dst_to_ring_current_energy(dst)\n    ratio = abs(dst) / B0 * 1.5\n    print(f"{name:40s} {dst:10d} {energy*1e-15:12.2f} {ratio:10.6f}")\n\nprint(f"\\nEm (dipole energy) = {Em_joules:.2e} J = {Em_joules*1e-15:.0f} PJ")\nprint(f"Even a super-intense storm uses only ~{300/31000*1.5*100:.3f}% of dipole energy")\nprint(f"초강력 폭풍도 쌍극자 에너지의 약 {300/31000*1.5*100:.3f}%만 사용합니다")

In [ ]:
# --- Visualize DPS relation ---\ndst_range = np.linspace(-600, 0, 200)\nenergy_pj = np.array([dst_to_ring_current_energy(d) * 1e-15 for d in dst_range])\n\nfig, ax = plt.subplots(figsize=(10, 7))\nax.plot(dst_range, energy_pj, "b-", linewidth=2)\n\n# Mark storm categories\ncategories = [\n    (-30, "Weak\\n약", "green"),\n    (-50, "Moderate\\n중", "orange"),\n    (-100, "Intense\\n강", "red"),\n    (-250, "Super-intense\\n초강", "darkred"),\n]\nfor dst_val, label, color in categories:\n    e = dst_to_ring_current_energy(dst_val) * 1e-15\n    ax.axvline(dst_val, color=color, linestyle="--", alpha=0.5)\n    ax.plot(dst_val, e, "o", color=color, markersize=10, zorder=5)\n    ax.annotate(label, (dst_val, e), textcoords="offset points",\n                xytext=(15, 10), fontsize=10, color=color,\n                fontweight="bold")\n\n# Mark historical events\nhistorical = [\n    (-589, "Quebec 1989", "purple"),\n    (-422, "Halloween 2003", "brown"),\n]\nfor dst_val, label, color in historical:\n    e = dst_to_ring_current_energy(dst_val) * 1e-15\n    ax.plot(dst_val, e, "s", color=color, markersize=10, zorder=5)\n    ax.annotate(label, (dst_val, e), textcoords="offset points",\n                xytext=(10, -20), fontsize=9, color=color)\n\nax.set_xlabel("$Dst^*$ [nT]")\nax.set_ylabel("Ring Current Energy [PJ]")\nax.set_title("DPS Relation: Dst ↔ Ring Current Energy\\n"\n             "DPS 관계: Dst ↔ Ring Current 에너지")\nax.invert_xaxis()\n\nplt.tight_layout()\nplt.savefig("15_gonzalez_1994_fig4_dps_relation.png", dpi=150, bbox_inches="tight")\nplt.show()

## 4. Storm Classification Diagram / 폭풍 분류 다이어그램\n\nVisualize the Gonzalez et al. storm classification scheme with the statistics they reported (1975–1986).\n논문에서 보고한 통계(1975–1986)를 기반으로 폭풍 분류 체계를 시각화합니다.

In [ ]:
# --- Storm classification visualization ---\nfig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))\n\n# Left: Classification bar chart\ncats = ["Weak\\n약\\n(-30 to -50)", "Moderate\\n중\\n(-50 to -100)",\n        "Intense\\n강\\n(< -100)", "Super-intense\\n초강\\n(< -250)"]\npercentages = [25, 63, 11, 1.5]  # From the paper\ncolors = ["#4CAF50", "#FF9800", "#F44336", "#880E4F"]\n\nbars = ax1.bar(cats, percentages, color=colors, edgecolor="black", linewidth=0.5)\nfor bar, pct in zip(bars, percentages):\n    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,\n             f"{pct}%", ha="center", fontsize=14, fontweight="bold")\n\nax1.set_ylabel("Fraction of All Storms [%]\\n전체 폭풍 비율 [%]")\nax1.set_title("Storm Classification Distribution (1975–1986)\\n"\n              "폭풍 분류 분포 (1975–1986)")\nax1.set_ylim(0, 75)\n\n# Right: Dst scale with thresholds\ndst_values = np.linspace(-500, 50, 300)\nax2.barh(0, 1, left=0, height=50, color="#E8F5E9", label="Quiet (> 0 nT)")  # dummy\n\n# Create colored Dst scale\nfor i in range(len(dst_values) - 1):\n    d = dst_values[i]\n    if d > -30:\n        color = "#E8F5E9\"  # Quiet - light green\n    elif d > -50:\n        color = "#4CAF50\"  # Weak - green\n    elif d > -100:\n        color = "#FF9800"  # Moderate - orange\n    elif d > -250:\n        color = "#F44336"  # Intense - red\n    else:\n        color = "#880E4F"  # Super-intense - dark red\n    ax2.axhspan(dst_values[i], dst_values[i+1], color=color, alpha=0.7)\n\n# Threshold lines and labels\nthresholds = [\n    (-30, "Weak threshold / 약 임계값"),\n    (-50, "Moderate threshold / 중 임계값"),\n    (-100, "Intense threshold / 강 임계값 (Gonzalez et al.)"),\n    (-250, "Super-intense / 초강 임계값"),\n]\nfor val, label in thresholds:\n    ax2.axhline(val, color="black", linewidth=2, linestyle="--")\n    ax2.text(0.55, val + 5, label, fontsize=10, fontweight="bold",\n             transform=ax2.get_yaxis_transform())\n\n# Historical markers\nhist_storms = [\n    (-589, "Quebec 1989\\n(-589 nT)"),\n    (-422, "Halloween 2003\\n(-422 nT)"),\n    (-100, "~1/month\\n(~월 1회)"),\n]\nfor val, label in hist_storms:\n    if val < -250:\n        ax2.annotate(label, (0.2, val), fontsize=9, color="white",\n                     fontweight="bold")\n\nax2.set_xlim(0, 1)\nax2.set_ylim(-500, 50)\nax2.set_ylabel("Dst [nT]")\nax2.set_title("Dst Storm Intensity Scale\\nDst 폭풍 강도 척도")\nax2.set_xticks([])\n\nplt.tight_layout()\nplt.savefig("15_gonzalez_1994_fig5_classification.png", dpi=150, bbox_inches="tight")\nplt.show()

## 5. Seasonal Distribution — Russell-McPherron Effect / 계절 분포 — Russell-McPherron 효과\n\nThe paper notes that intense storms show seasonal variation peaking at equinoxes (March and September), consistent with the Russell-McPherron [1973] effect. We reproduce Figure 6's seasonal distribution pattern.\n\n논문은 intense storm이 춘분과 추분(3월, 9월)에 극대를 보이는 계절 변동을 지적합니다. Figure 6의 계절 분포 패턴을 재현합니다.

In [ ]:
# --- Seasonal distribution of intense storms (reproducing Figure 6 pattern) ---\n# Data digitized from Figure 6 (Gonzalez and Tsurutani [1992])\n# Normalized number of intense storms per month, 1975-1986\nmonths = np.arange(1, 13)\nmonth_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",\n               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]\n\n# Approximate values from Figure 6 in the paper\nstorms_per_month = np.array([6, 7, 12, 10, 5, 3, 4, 5, 11, 13, 8, 4])\n\n# Russell-McPherron model: equinoctial modulation\n# The RM effect predicts maxima near equinoxes due to favorable\n# geometry for southward Bz projection in GSM coordinates\ntheta_tilt = 23.4 * np.pi / 180  # Earth's axial tilt\nday_of_year = np.array([15, 45, 75, 105, 135, 165, 195, 225, 255, 285, 315, 345])\n# Simplified RM effect: sin^2 of angle between GSM z and solar wind B\nrm_factor = np.cos(2 * np.pi * (day_of_year - 80) / 365)**2  # Peak at equinoxes\nrm_normalized = rm_factor / rm_factor.max() * storms_per_month.max()\n\nfig, ax = plt.subplots(figsize=(12, 6))\n\nbars = ax.bar(months, storms_per_month, color="#F44336", edgecolor="black",\n              alpha=0.8, width=0.7, label="Intense storms (Dst < -100 nT)\\nfrom Gonzalez & Tsurutani [1992]")\nax.plot(months, rm_normalized, "b-o", linewidth=2, markersize=8,\n        label="Russell-McPherron modulation\\n$\\\\cos^2(2\\\\pi(d-80)/365)$")\n\n# Annotate equinoxes and solstices\nax.axvline(3, color="green", linestyle="--", alpha=0.3)\nax.axvline(9, color="green", linestyle="--", alpha=0.3)\nax.text(3, storms_per_month.max() * 1.05, "Vernal Equinox\\n춘분",\n        ha="center", fontsize=10, color="green")\nax.text(9, storms_per_month.max() * 1.05, "Autumnal Equinox\\n추분",\n        ha="center", fontsize=10, color="green")\n\nax.set_xticks(months)\nax.set_xticklabels(month_names)\nax.set_xlabel("Month / 월")\nax.set_ylabel("Normalized Number of Intense Storms\\n정규화된 강한 폭풍 수")\nax.set_title("Seasonal Distribution of Intense Storms (1975–1986)\\n"\n             "강한 폭풍의 계절 분포 (1975–1986) — cf. Figure 6")\nax.legend(loc="upper right")\nax.set_ylim(0, storms_per_month.max() * 1.2)\n\nplt.tight_layout()\nplt.savefig("15_gonzalez_1994_fig6_seasonal.png", dpi=150, bbox_inches="tight")\nplt.show()

## 6. Storm vs Substorm: Conceptual Diagram / 폭풍 vs 서브스톰: 개념도\n\nReproducing the key conceptual distinction from the paper (cf. Figure 11):\n논문의 핵심 개념적 구분을 재현합니다 (Figure 11 참조):\n\n- **Substorm**: Magnetotail energy loading/unloading (~1–3 h) / 자기꼬리 에너지 축적·방출\n- **HILDCAA**: High-intensity long-duration continuous AE activity / 고강도 장기 지속 AE 활동\n- **Storm**: Ring current enhancement via sustained southward $B_z$ / 지속 남향 $B_z$에 의한 환전류 강화

In [ ]:
# --- Storm vs Substorm comparison (synthetic time series) ---\nfig, axes = plt.subplots(3, 2, figsize=(16, 12))\n\nt_sub = np.linspace(0, 12, 500)  # 12 hours\nt_storm = np.linspace(0, 72, 500)  # 72 hours\n\n# === Left column: Isolated Substorm ===\n# Bz: brief southward turning\nbz_sub = 3 + 2 * np.sin(2 * np.pi * t_sub / 4)\nbz_sub[(t_sub > 3) & (t_sub < 5)] = -5\nbz_sub[(t_sub > 7) & (t_sub < 8.5)] = -4\n\n# AE: substorm signatures\nae_sub = np.full_like(t_sub, 50.0)\nae_sub[(t_sub > 3.5) & (t_sub < 6)] = 800 * np.exp(-((t_sub[(t_sub > 3.5) & (t_sub < 6)] - 4.2)**2) / 0.3)\nae_sub[(t_sub > 7.5) & (t_sub < 9.5)] = 500 * np.exp(-((t_sub[(t_sub > 7.5) & (t_sub < 9.5)] - 8.2)**2) / 0.3)\n\n# Dst: minimal response\ndst_sub = np.full_like(t_sub, -5.0)\ndst_sub[(t_sub > 4) & (t_sub < 7)] = -20\ndst_sub[(t_sub > 8) & (t_sub < 10)] = -15\n\naxes[0, 0].plot(t_sub, bz_sub, "r-", linewidth=1.5)\naxes[0, 0].fill_between(t_sub, bz_sub, 0, where=(bz_sub < 0), alpha=0.3, color="red")\naxes[0, 0].axhline(0, color="k", linewidth=0.5)\naxes[0, 0].set_ylabel("IMF $B_z$ [nT]")\naxes[0, 0].set_title("Isolated Substorm / 고립 서브스톰")\n\naxes[1, 0].plot(t_sub, ae_sub, "g-", linewidth=1.5)\naxes[1, 0].set_ylabel("AE [nT]")\naxes[1, 0].set_ylim(0, 1500)\n\naxes[2, 0].plot(t_sub, dst_sub, "b-", linewidth=2)\naxes[2, 0].axhline(-30, color="orange", linestyle=":", alpha=0.5)\naxes[2, 0].set_ylabel("Dst [nT]")\naxes[2, 0].set_xlabel("Time [hours]")\naxes[2, 0].set_ylim(-200, 20)\naxes[2, 0].text(6, -180, "Dst > -30 nT\\nNOT a storm\\n폭풍 아님",\n                fontsize=12, color="green", fontweight="bold",\n                bbox=dict(facecolor="white", alpha=0.8))\n\n# === Right column: Magnetic Storm ===\n# Bz: sustained southward\nbz_storm = np.full_like(t_storm, 3.0)\nbz_storm[(t_storm > 2) & (t_storm < 8)] = -8 + 5 * np.random.randn(((t_storm > 2) & (t_storm < 8)).sum()) * 0.3\nbz_storm[(t_storm > 8) & (t_storm < 30)] = -15 * np.sin(np.pi * (t_storm[(t_storm > 8) & (t_storm < 30)] - 8) / 22)\n\n# AE: multiple substorms embedded\nnp.random.seed(123)\nae_storm = np.full_like(t_storm, 100.0)\nfor peak_t in [5, 10, 14, 18, 22, 26, 32, 38]:\n    peak_amp = np.random.uniform(400, 1200)\n    ae_storm += peak_amp * np.exp(-((t_storm - peak_t)**2) / 0.8)\n\n# Dst: strong depression\nbs_storm = compute_bs(bz_storm)\nq_storm = injection_rate(np.full_like(t_storm, 500.0), bs_storm)\nsol_storm = solve_ivp(\n    burton_equation, [t_storm[0], t_storm[-1]], [0.0],\n    args=(t_storm, q_storm, 7.7), t_eval=t_storm, method="RK45", max_step=0.1\n)\ndst_storm = sol_storm.y[0]\n\naxes[0, 1].plot(t_storm, bz_storm, "r-", linewidth=1)\naxes[0, 1].fill_between(t_storm, bz_storm, 0, where=(bz_storm < 0), alpha=0.3, color="red")\naxes[0, 1].axhline(0, color="k", linewidth=0.5)\naxes[0, 1].set_ylabel("IMF $B_z$ [nT]")\naxes[0, 1].set_title("Magnetic Storm / 자기 폭풍")\n\naxes[1, 1].plot(t_storm, ae_storm, "g-", linewidth=1)\naxes[1, 1].set_ylabel("AE [nT]")\naxes[1, 1].set_ylim(0, 1500)\naxes[1, 1].text(15, 1300, "Multiple substorms\\n다중 서브스톰",\n                fontsize=10, color="green")\n\naxes[2, 1].plot(t_storm, dst_storm, "b-", linewidth=2)\naxes[2, 1].axhline(-30, color="orange", linestyle=":", alpha=0.5)\naxes[2, 1].axhline(-100, color="red", linestyle=":", alpha=0.5,\n                    label="Intense threshold")\naxes[2, 1].set_ylabel("Dst [nT]")\naxes[2, 1].set_xlabel("Time [hours]")\naxes[2, 1].set_ylim(-200, 20)\naxes[2, 1].text(40, -180, f"Peak Dst = {dst_storm.min():.0f} nT\\nINTENSE STORM\\n강한 폭풍",\n                fontsize=12, color="red", fontweight="bold",\n                bbox=dict(facecolor="white", alpha=0.8))\naxes[2, 1].legend(loc="lower left")\n\nfig.suptitle("Storm ≠ Substorm: The Key Distinction (Gonzalez et al., 1994)\\n"\n             "폭풍 ≠ 서브스톰: 핵심 구별 (Gonzalez et al., 1994)",\n             fontsize=14, fontweight="bold", y=1.01)\n\nplt.tight_layout()\nplt.savefig("15_gonzalez_1994_fig7_storm_vs_substorm.png", dpi=150, bbox_inches="tight")\nplt.show()

## Summary / 요약\n\nThis notebook implemented the key quantitative frameworks from Gonzalez et al. (1994):\n이 노트북은 Gonzalez et al. (1994)의 핵심 정량적 프레임워크를 구현했습니다:\n\n1. **Burton Equation** — Simulated a magnetic cloud storm with realistic solar wind inputs, showing Dst evolution through all storm phases (SSC → main phase → recovery). Demonstrated that the model reproduces the characteristic storm profile.\n   Burton 방정식을 사용하여 현실적 태양풍 입력으로 magnetic cloud 폭풍을 시뮬레이션하고, 모든 폭풍 단계의 Dst 진화를 보여주었습니다.\n\n2. **Decay Time Sensitivity** — Showed how $\\tau$ critically affects both storm intensity and recovery rate. Smaller $\\tau$ produces deeper storms but faster recovery.\n   $\\tau$가 폭풍 강도와 회복 속도에 미치는 영향을 보여주었습니다.\n\n3. **Coupling Functions** — Compared $VB_s$, $\\epsilon$, and $VB_T\\sin^2(\\theta/2)$ for the same storm, showing their different sensitivities to IMF geometry.\n   동일 폭풍에 대해 다양한 결합 함수를 비교했습니다.\n\n4. **DPS Relation** — Quantified ring current energy for various storm intensities, from weak to Carrington-class events.\n   DPS 관계를 사용하여 다양한 폭풍 강도의 ring current 에너지를 정량화했습니다.\n\n5. **Storm Classification & Seasonal Distribution** — Visualized the Dst-based classification scheme and the Russell-McPherron equinoctial modulation.\n   Dst 기반 분류 체계와 Russell-McPherron 춘추분 변동을 시각화했습니다.\n\n6. **Storm vs Substorm** — Illustrated the fundamental distinction with synthetic time series.\n   합성 시계열로 폭풍과 서브스톰의 근본적 차이를 보여주었습니다.